# ⚡ PINN for Series RLC Circuit
> **Physics-Informed Neural Networks** — solving and identifying circuit parameters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/pinn-rlc-circuit/blob/main/notebooks/walkthrough.ipynb)

---
This notebook walks through all 4 phases of the project:
- **Phase 1** — Physics & math
- **Phase 2** — Build the PINN
- **Phase 3** — Validate across damping regimes
- **Phase 4** — Inverse problem: discover R, L, C from noisy data

In [ ]:
# Install dependencies (only needed on Colab)
!pip install torch numpy scipy matplotlib -q

## Phase 1 — The Physics

The series RLC circuit obeys Kirchhoff's Voltage Law:

$$L\frac{d^2q}{dt^2} + R\frac{dq}{dt} + \frac{q}{C} = 0$$

The **damping ratio** $\zeta = \frac{R}{2}\sqrt{\frac{C}{L}}$ determines behavior:
- $\zeta < 1$ → underdamped (oscillates)
- $\zeta = 1$ → critically damped
- $\zeta > 1$ → overdamped

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Circuit parameters
R, L, C = 10.0, 1.0, 0.01
zeta = (R / 2) * (C / L) ** 0.5
omega0 = 1.0 / (L * C) ** 0.5
print(f'ω₀ = {omega0:.2f} rad/s')
print(f'ζ  = {zeta:.3f} → {"Underdamped" if zeta < 1 else "Overdamped"}')

## Phase 2 — Build the PINN

The network maps $t \rightarrow \hat{q}(t)$ and is trained to satisfy the ODE.

In [ ]:
class PINN(nn.Module):
    def __init__(self, hidden=64, n_layers=4):
        super().__init__()
        layers = [nn.Linear(1, hidden), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*layers)
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, t):
        return self.net(t)

def physics_loss(model, t_phys, R, L, C):
    t_phys = t_phys.requires_grad_(True)
    q = model(t_phys)
    q_t  = torch.autograd.grad(q,   t_phys, grad_outputs=torch.ones_like(q),   create_graph=True)[0]
    q_tt = torch.autograd.grad(q_t, t_phys, grad_outputs=torch.ones_like(q_t), create_graph=True)[0]
    return torch.mean((L * q_tt + R * q_t + q / C) ** 2)

def ic_loss(model):
    t0 = torch.tensor([[0.0]], requires_grad=True)
    q0 = model(t0)
    q0_t = torch.autograd.grad(q0, t0, grad_outputs=torch.ones_like(q0), create_graph=True)[0]
    return (q0 - 1.0)**2 + q0_t**2

print('Network and loss functions defined!')

In [ ]:
torch.manual_seed(42)
model = PINN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
t_phys = torch.FloatTensor(1500, 1).uniform_(0.0, 2.0)

print(f'{"Epoch":>7} | {"Loss":>10}')
for epoch in range(1, 12001):
    optimizer.zero_grad()
    loss = physics_loss(model, t_phys, R, L, C) + 100.0 * ic_loss(model)
    loss.backward()
    optimizer.step()
    if epoch % 2000 == 0:
        print(f'{epoch:>7} | {loss.item():>10.6f}')

print('Training complete!')

In [ ]:
# Compare PINN vs scipy
def rlc_ref(t, y): return [y[1], (-R*y[1] - y[0]/C)/L]
sol = solve_ivp(rlc_ref, [0,2], [1.0, 0.0], t_eval=np.linspace(0,2,500), method='RK45', rtol=1e-9)

t_test = torch.linspace(0, 2, 500).reshape(-1,1)
with torch.no_grad():
    q_pred = model(t_test).numpy().flatten()

plt.figure(figsize=(9,4))
plt.plot(sol.t, sol.y[0], 'k-', lw=2.5, label='ODE Solver (ground truth)')
plt.plot(t_test.numpy(), q_pred, 'r--', lw=2, label='PINN Prediction')
plt.xlabel('Time (s)'); plt.ylabel('q(t) [C]')
plt.title(f'PINN vs ODE Solver — R={R}Ω, L={L}H, C={C}F')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f'Max error: {np.max(np.abs(q_pred - np.interp(t_test.numpy().flatten(), sol.t, sol.y[0]))):.5f} C')

## Phase 3 — Validation
Run `src/phase3_validation.py` for full validation across all three damping regimes.

## Phase 4 — Inverse Problem
Run `src/phase4_inverse.py` to discover unknown R, L, C from noisy measurements.

Key insight: we learn $\alpha = R/L$ and $\omega^2 = 1/LC$ (the identifiable quantities), then reconstruct R and C using L as an anchor.

---
## 📚 References
- Raissi et al. (2019) — Physics-informed neural networks, *Journal of Computational Physics*
- [Project GitHub](https://github.com/YOUR_USERNAME/pinn-rlc-circuit)